# Simple GenAI App


In [38]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"

os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [39]:
# Data Ingestion -  From the website we'll scrape the data
from langchain_community.document_loaders import WebBaseLoader


In [40]:
# using loader

loader = WebBaseLoader(web_path="https://docs.langchain.com/oss/python/langchain/observability")

loader

In [41]:
docs = loader.load()

docs

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langchain/observability', 'title': 'LangSmith Observability - Docs by LangChain', 'language': 'en'}, page_content='LangSmith Observability - Docs by LangChainSkip to main contentDocs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationDeploy with LangSmithLangSmith ObservabilityDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIDeploy with LangSmithDeploymentObservabilityOn this pagePrerequisitesEnable tracingQuickstartTrace selectivelyLog to a projectAdd metadata to tracesDeploy with

In [ ]:
# Load Data ---> Docs ---> Divide the document into chunks ---> vectors ---> vector embeddings ---> store in vectoredb

# Divide the documents into chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
documents = text_splitter.split_documents(docs)

documents

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langchain/observability', 'title': 'LangSmith Observability - Docs by LangChain', 'language': 'en'}, page_content='LangSmith Observability - Docs by LangChainSkip to main contentDocs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationDeploy with LangSmithLangSmith ObservabilityDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIDeploy with LangSmithDeploymentObservabilityOn this pagePrerequisitesEnable tracingQuickstartTrace selectivelyLog to a projectAdd metadata to tracesDeploy with

In [43]:
# converting into vectors

from langchain_openai import OpenAIEmbeddings
embedding = OpenAIEmbeddings()

In [44]:
from langchain_community.vectorstores import FAISS

vectorStoreDB = FAISS.from_documents(documents, embedding)

In [45]:
vectorStoreDB

In [46]:
# Query from vectorstoreDB

query = " including all tool calls, model interactions, and decision points."
result = vectorStoreDB.similarity_search(query)
result


[Document(id='03d58bc6-35ac-4c4b-a984-7fd6330e9985', metadata={'source': 'https://docs.langchain.com/oss/python/langchain/observability', 'title': 'LangSmith Observability - Docs by LangChain', 'language': 'en'}, page_content='Traces record every step of your agent’s execution, from the initial user input to the final response, including all tool calls, model interactions, and decision points. This execution data helps you debug issues, evaluate performance across different inputs, and monitor usage patterns in production.\nThis guide shows you how to enable tracing for your LangChain agents and use LangSmith to analyze their execution.\n\u200bPrerequisites\nBefore you begin, ensure you have the following:'),
 Document(id='aa13b08b-7c7a-48fb-a3c8-0c2d15f0a1a6', metadata={'source': 'https://docs.langchain.com/oss/python/langchain/observability', 'title': 'LangSmith Observability - Docs by LangChain', 'language': 'en'}, page_content='to tracesDeploy with LangSmithLangSmith ObservabilityC

In [47]:
result[0].page_content

'Traces record every step of your agent’s execution, from the initial user input to the final response, including all tool calls, model interactions, and decision points. This execution data helps you debug issues, evaluate performance across different inputs, and monitor usage patterns in production.\nThis guide shows you how to enable tracing for your LangChain agents and use LangSmith to analyze their execution.\n\u200bPrerequisites\nBefore you begin, ensure you have the following:'

In [48]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o"
)

print(llm)

profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x00000296E6376770> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000296E63766E0> root_client=<openai.OpenAI object at 0x00000296E6376560> root_async_client=<openai.AsyncOpenAI object at 0x00000296E6377A60> model_name='gpt-4o' model_kwargs={} openai_api_key=SecretStr('**********') stream_usage=True


In [49]:
# Retrieval Chain, Document Chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_template(
    """

Answer the following question based only on the provided context:
<context>
{context}
</context>

Keep the provided input strictly in consideration:
<input>
{input}
</input>

"""
)

document_chain = create_stuff_documents_chain(llm, prompt)

document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\nKeep the provided input strictly in consideration:\n<input>\n{input}\n</input>\n\n'), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': Tru

In [50]:
from langchain_core.documents import Document

document_chain.invoke({
    "input":"Why do we need visibility in langchain?",
    "context":[Document(page_content="As you build and run agents with LangChain, you need visibility into how they behave: which tools they call, what prompts they generate, and how they make decisions. LangChain agents built with create_agent automatically support tracing through LangSmith, a platform for capturing, debugging, evaluating, and monitoring LLM application behavior.Traces record every step of your agents execution, from the initial user input to the final response, including all tool calls, model interactions, and decision points. This execution data helps you debug issues, evaluate performance across different inputs, and monitor usage patterns in production.")]
    })

"We need visibility in LangChain to understand how the agents behave, specifically which tools they call, what prompts they generate, and how they make decisions. This visibility allows us to trace every step of the agent's execution, helping in debugging issues, evaluating performance across different inputs, and monitoring usage patterns in production."

We want the documents to first come from the retriever we just set up. THat way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [51]:
# Retriever ---  Can be considered as a interface
# Input ---> Retriever --->VectorStoreDB

vectorStoreDB

In [52]:
retriever = vectorStoreDB.as_retriever()
from langchain_classic.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retriever,document_chain)
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000296E6377C70>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\nKeep the provided input 

In [54]:
# Get the response from the LLM

response = retrieval_chain.invoke({"input":"Why do we need visibility in langchain agents through langsmith?"})

In [55]:
response['answer']

"Visibility in LangChain agents through LangSmith is needed to understand how these agents behave. This includes knowing which tools they call, what prompts they generate, and how they make decisions. LangSmith provides tracing that records every step of an agent's execution. This execution data helps in debugging issues, evaluating performance across different inputs, and monitoring usage patterns in production."

In [57]:
response['context']

[Document(id='aa13b08b-7c7a-48fb-a3c8-0c2d15f0a1a6', metadata={'source': 'https://docs.langchain.com/oss/python/langchain/observability', 'title': 'LangSmith Observability - Docs by LangChain', 'language': 'en'}, page_content='to tracesDeploy with LangSmithLangSmith ObservabilityCopy pageCopy pageAs you build and run agents with LangChain, you need visibility into how they behave: which tools they call, what prompts they generate, and how they make decisions. LangChain agents built with create_agent automatically support tracing through LangSmith, a platform for capturing, debugging, evaluating, and monitoring LLM application behavior.'),
 Document(id='ccd16f28-f03a-4c44-8fa3-c3a39c39cdac', metadata={'source': 'https://docs.langchain.com/oss/python/langchain/observability', 'title': 'LangSmith Observability - Docs by LangChain', 'language': 'en'}, page_content='LangSmith Observability - Docs by LangChainSkip to main contentDocs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTr